# 02 — Dataset Inspection & Exploratory Data Analysis (EDA)
### AI Interview System — 100% Project-Owned ML Pipeline
This notebook performs exploratory data analysis using **Pandas & Seaborn**:
1. **`data.head()`**: Preview sample dataset rows and feature columns.
2. **`data.info()`**: Inspect schema types, non-null counts, and memory footprint.
3. **`data.describe()`**: Compute statistical distributions (mean, std, min, quartiles, max).
4. **`data.isnull().sum()`**: Tally missing values across all columns.
5. **`sns.heatmap(corr_matrix)`**: Visualize feature correlations (question length, answer length, difficulty, domain).
6. **Class & Domain Distribution**: Plot category breakdowns across difficulty tiers and technical domains.


In [ ]:
# Cell 1: Environment Setup & Load DataFrame (`data`)
import os
import sys
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Auto-detect workspace root (supports Google Colab, local terminal, or notebooks/ subfolder)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    if Path('/content/ai-interview-system/ml-service').exists():
        WORKSPACE_DIR = Path('/content/ai-interview-system/ml-service')
    elif Path('/content/drive/MyDrive/ai-interview-system/ml-service').exists():
        WORKSPACE_DIR = Path('/content/drive/MyDrive/ai-interview-system/ml-service')
    else:
        WORKSPACE_DIR = Path(os.getcwd())
    print("[OK] Running in Google Colab:", WORKSPACE_DIR)
except ImportError:
    cwd = Path(os.getcwd())
    if cwd.name == "notebooks":
        WORKSPACE_DIR = cwd.parent
    elif (cwd / "ml-service").exists():
        WORKSPACE_DIR = cwd / "ml-service"
    else:
        WORKSPACE_DIR = cwd
    print("[OK] Running in local environment:", WORKSPACE_DIR)

WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE_DIR)
if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))
print(f"[OK] Working Directory set to: {WORKSPACE_DIR}")

RAW_DATASET_FILE = WORKSPACE_DIR / "dataset" / "raw" / "raw_interview_dataset.json"
FIGURES_EDA_DIR = WORKSPACE_DIR / "reports" / "figures" / "eda"
FIGURES_EDA_DIR.mkdir(parents=True, exist_ok=True)

with open(RAW_DATASET_FILE, "r", encoding="utf-8") as f:
    records = json.load(f)

# Load into Pandas DataFrame
data = pd.DataFrame(records)

# Feature engineering: compute text metrics and encoded numerical representations
data["q_char_len"] = data["question"].fillna("").astype(str).str.len()
data["q_word_count"] = data["question"].fillna("").astype(str).str.split().str.len()
data["a_char_len"] = data["answer"].fillna("").astype(str).str.len()
data["a_word_count"] = data["answer"].fillna("").astype(str).str.split().str.len()
data["difficulty_encoded"] = data["difficulty"].map({"Beginner": 1, "Intermediate": 2, "Advanced": 3}).fillna(2).astype(int)
data["domain_encoded"] = data["domain"].astype("category").cat.codes

print(f"Loaded DataFrame with shape: {data.shape} ({data.shape[0]} rows, {data.shape[1]} columns)")


In [ ]:
# Cell 2: data.head() — Dataset Preview
print("=== FIRST 5 ROWS (data.head()) ===")
data.head()


In [ ]:
# Cell 3: data.info() — Dataset Schema & Data Types
print("=== DATASET INFO (data.info()) ===")
data.info()


In [ ]:
# Cell 4: data.describe() — Statistical Summary
print("=== DESCRIPTIVE STATISTICS (data.describe()) ===")
data[["q_char_len", "q_word_count", "a_char_len", "a_word_count", "difficulty_encoded", "domain_encoded"]].describe()


In [ ]:
# Cell 5: data.isnull().sum() — Missing Values Check
print("=== MISSING VALUES COUNT (data.isnull().sum()) ===")
print(data.isnull().sum())


In [ ]:
# Cell 6: Correlation Matrix & Heatmap
corr_metrix = data[["q_char_len", "q_word_count", "a_char_len", "a_word_count", "difficulty_encoded", "domain_encoded"]].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_metrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5, cbar=True)
plt.title("Correlation Heat Map", fontsize=14, fontweight="bold", pad=12)
plt.tight_layout()

# Save figure to reports/figures/eda/
heat_path = FIGURES_EDA_DIR / "correlation_heatmap.png"
plt.savefig(heat_path, dpi=300)
print(f"Saved correlation heatmap to: {heat_path}")
plt.show()


In [ ]:
# Cell 7: Category & Difficulty Distribution Plots
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Difficulty distribution
sns.countplot(data=data, x="difficulty", order=["Beginner", "Intermediate", "Advanced"], palette="coolwarm", ax=axes[0], hue="difficulty", legend=False)
axes[0].set_title("Difficulty Level Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Difficulty")
axes[0].set_ylabel("Count")

# Domain distribution
domain_order = data["domain"].value_counts().index
sns.countplot(data=data, y="domain", order=domain_order, palette="mako", ax=axes[1], hue="domain", legend=False)
axes[1].set_title("Domain Category Breakdown", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Number of Questions")
axes[1].set_ylabel("Domain")

plt.tight_layout()
dist_path = FIGURES_EDA_DIR / "class_distribution.png"
plt.savefig(dist_path, dpi=300)
print(f"Saved class distribution plot to: {dist_path}")
plt.show()

print("Stage 02 (EDA & Dataset Inspection) Completed Successfully.")
